In [ ]:
import os
from dotenv import load_dotenv
from litellm import completion

# 1. Φόρτωση API Key από το .env αρχείο
load_dotenv(override=True)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("❌ OPENAI_API_KEY is missing. Check your .env file!")

# 2. DATA SOURCE (Retrieval Step)
# Σε ένα πλήρες RAG, αυτά τα δεδομένα θα προέρχονταν από Vector DB ή PDF parsing.
knowledge_base_document = """
CLINVAR VARIANT REPORT:
- Variant ID: 17661
- Gene: BRCA1
- HGVS: c.5266dupC (p.Gln1756ProfsTer74)
- Clinical Significance: Pathogenic
- Condition: Hereditary Breast and Ovarian Cancer Syndrome
- Review Status: Reviewed by expert panel
- Summary: This sequence change variation consists of an insertion of a single nucleotide (C) 
  at position 5266, causing a frameshift mutation that leads to a premature stop codon 74 amino acids downstream.
"""

# 3. RAG Core Function
def answer_query_with_rag(user_question: str, context_document: str):
    """
    Augmented Generation:
    Εμπλουτίζουμε το prompt περνώντας το κείμενο ως context (Retrieval)
    και ζητάμε από το LLM να απαντήσει αυστηρά βάσει αυτού.
    """
    
    system_prompt = """
    You are an expert Clinical Bioinformatics Assistant.
    Answer the user's question USING ONLY the provided Knowledge Base context.
    If the answer cannot be found in the context, explicitly state "Information not available in context."
    Do not use external knowledge.
    """

    user_prompt = f"""
    --- KNOWLEDGE BASE CONTEXT ---
    {context_document}
    ------------------------------

    USER QUESTION: {user_question}
    """

    print(f"❓ Question: {user_question}\n")
    print("🤖 Querying LLM via LiteLLM...")

    # Κλήση του μοντέλου μέσω LiteLLM
    response = completion(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0  # Χαμηλή θερμοκρασία για ακρίβεια χωρίς δημιουργικότητα
    )

    return response.choices[0].message.content


# --- Execution Block ---
if __name__ == "__main__":
    # Ερώτηση 1: Υπάρχει στο Document
    q1 = "What is the clinical significance and gene associated with variant 17661?"
    answer1 = answer_query_with_rag(q1, knowledge_base_document)
    print(f"\n💡 Answer:\n{answer1}\n")
    print("=" * 60 + "\n")

    # Ερώτηση 2: ΔΕΝ υπάρχει στο Document (Έλεγχος για Hallucinations)
    q2 = "What is the recommended medication dosage for this patient?"
    answer2 = answer_query_with_rag(q2, knowledge_base_document)
    print(f"\n💡 Answer:\n{answer2}\n")